In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

!unzip /content/drive/MyDrive/coco_total.zip -d /content


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: /content/coco_total/labels/train2017/000000039711.txt  
  inflating: /content/coco_total/labels/train2017/000000261879.txt  
  inflating: /content/coco_total/labels/train2017/000000336404.txt  
  inflating: /content/coco_total/labels/train2017/000000007095.txt  
  inflating: /content/coco_total/labels/train2017/126_jpg.rf.45a6f7659d71e11ef3e4585143fa8986.txt  
  inflating: /content/coco_total/labels/train2017/000000224118.txt  
  inflating: /content/coco_total/labels/train2017/000000019176.txt  
  inflating: /content/coco_total/labels/train2017/000000278867.txt  
  inflating: /content/coco_total/labels/train2017/000000511200.txt  
  inflating: /content/coco_total/labels/train2017/000000431241.txt  
  inflating: /content/coco_total/labels/train2017/000000152397.txt  
  inflating: /content/coco_total/labels/train2017/000000166630.txt  
  inflating: /content/coco_total/labels/train2017/000000273446.txt  
  inflating: /content/coco_total/lab

In [ ]:
import os

# Check if the unzipping was successful and files exist
if not os.path.exists('/content/coco_total/data.yaml'):
    print("Error: /content/coco_total/data.yaml not found. Please ensure the dataset was unzipped correctly.")
else:
    print("Dataset directory and data.yaml found. Proceeding with training preparation.")

Dataset directory and data.yaml found. Proceeding with training preparation.


In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 89.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Load the pre-trained model from /content/last.pt
# 'last.pt' usually refers to the last trained weights from a previous YOLO run.
# If this is a detection model, it will be loaded as such.
model = YOLO('/content/last.pt')

print("Model loaded successfully from /content/last.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model loaded successfully from /content/last.pt


In [ ]:
import os
from collections import defaultdict

label_dirs = {
    "train2017": "/content/coco_total/labels/train2017",
    "val2017": "/content/coco_total/labels/val2017",
    "test2017": "/content/coco_total/labels/test2017"
}

all_class_counts = defaultdict(lambda: defaultdict(int))

for split_name, label_dir in label_dirs.items():
    if not os.path.exists(label_dir):
        print(f"Warning: Label directory for {split_name} not found at {label_dir}")
        continue

    print(f"\nProcessing {split_name} labels in: {label_dir}")
    class_counts = defaultdict(int)
    total_objects = 0

    label_files = [f for f in os.listdir(label_dir) if f.endswith('.txt')]
    if not label_files:
        print(f"No label files found in {label_dir}")
        continue

    for label_file in label_files:
        file_path = os.path.join(label_dir, label_file)
        try:
            with open(file_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        class_id = int(parts[0])
                        class_counts[class_id] += 1
                        total_objects += 1
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    if total_objects > 0:
        print(f"Total objects in {split_name}: {total_objects}")
        print(f"Object counts by class ID for {split_name}:")
        for class_id, count in sorted(class_counts.items()):
            print(f"  Class {class_id}: {count}")
            all_class_counts[split_name][class_id] = count
    else:
        print(f"No objects found in {split_name} labels.")

# Optional: Provide a summary across all splits if needed
print("\n--- Overall Summary ---")
all_classes = sorted(list(set(cid for split_data in all_class_counts.values() for cid in split_data.keys())))

if all_classes:
    print(f"{'Class ID':<10}" + ''.join([f"{split_name:<15}" for split_name in label_dirs.keys() if os.path.exists(label_dirs[split_name])]))
    print("-" * (10 + 15 * len([s for s in label_dirs.keys() if os.path.exists(label_dirs[s])])))
    for class_id in all_classes:
        row = f"{class_id:<10}"
        for split_name in label_dirs.keys():
            if os.path.exists(label_dirs[split_name]):
                row += f"{all_class_counts[split_name].get(class_id, 0):<15}"
        print(row)
else:
    print("No classes found across any splits.")



Processing train2017 labels in: /content/coco_total/labels/train2017
Total objects in train2017: 369643
Object counts by class ID for train2017:
  Class 0: 257252
  Class 1: 7056
  Class 2: 43531
  Class 3: 8654
  Class 4: 6061
  Class 5: 9970
  Class 6: 12842
  Class 7: 1983
  Class 8: 9820
  Class 9: 5500
  Class 10: 3480
  Class 11: 2104
  Class 12: 1390

Processing val2017 labels in: /content/coco_total/labels/val2017
Total objects in val2017: 15744
Object counts by class ID for val2017:
  Class 0: 10777
  Class 1: 314
  Class 2: 1918
  Class 3: 367
  Class 4: 283
  Class 5: 414
  Class 6: 634
  Class 7: 75
  Class 8: 411
  Class 9: 218
  Class 10: 97
  Class 11: 185
  Class 12: 51

Processing test2017 labels in: /content/coco_total/labels/test2017
Total objects in test2017: 106
Object counts by class ID for test2017:
  Class 10: 7
  Class 11: 99

--- Overall Summary ---
Class ID  train2017      val2017        test2017       
-------------------------------------------------------

In [ ]:
import yaml
import os

# Path to the data.yaml file
yaml_path = '/content/coco_total/data.yaml'

# Read the existing yaml file
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Update paths to be absolute to avoid FileNotFoundError during training
    # We set the root 'path' and relative paths for train/val/test
    data['path'] = '/content/coco_total'
    data['train'] = 'images/train2017'
    data['val'] = 'images/val2017'
    data['test'] = 'images/test2017'

    # Save the updated yaml file
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)

    print(f"Successfully updated {yaml_path} with absolute paths.")
else:
    print(f"Error: {yaml_path} not found.")

Successfully updated /content/coco_total/data.yaml with absolute paths.


In [ ]:
from ultralytics import YOLO

# Load the model
model = YOLO('/content/last.pt')

# Train the model
# Using the fixed data.yaml
results = model.train(
    data='/content/coco_total/data.yaml',
    epochs=1,
    imgsz=640,
    batch=128,
    device=0,
    name='coco_total_yolo26n_train_fixed'
)

print("Training started...")

Ultralytics 8.4.15 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/coco_total/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_total_yolo26n_train_fixed, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [ ]:
from ultralytics import YOLO
import cv2
import os

# Define paths
model_path = '/content/runs/detect/coco_total_yolo26n_train_fixed/weights/best.pt'
image_path = '/content/100733546.s2.jpg'
output_path = '/content/100733546.s2_1.jpg'

# Check if model exists
if not os.path.exists(model_path):
    print(f"Error: Model weights not found at {model_path}. Please check if training finished successfully.")
else:
    # Load the trained model
    model = YOLO(model_path)

    # Run inference
    results = model(image_path)

    # Plot the results (returns a numpy array in BGR format)
    im_array = results[0].plot()

    # Save the image to the specified output path
    cv2.imwrite(output_path, im_array)
    print(f"Inference completed. Result saved to {output_path}")


image 1/1 /content/100733546.s2.jpg: 448x640 4 persons, 81.8ms
Speed: 1.2ms preprocess, 81.8ms inference, 0.3ms postprocess per image at shape (1, 3, 448, 640)
Inference completed. Result saved to /content/100733546.s2_1.jpg


In [ ]:
from ultralytics import YOLO

# Load the trained model
model = YOLO('/content/runs/detect/coco_total_yolo26n_train_fixed/weights/best.pt')

# Export the model to TFLite format
# This might take a few minutes and may install additional dependencies like onnx, onnx2tf, etc.
path = model.export(format='tflite')

print(f"Model exported successfully. Saved at: {path}")

Ultralytics 8.4.15 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon Platinum 8481C CPU @ 2.70GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,377,371 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/runs/detect/coco_total_yolo26n_train_fixed/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 18 packages in 6.58s
Prepared 9 packages in 3.37s
Installed 9 packages in 175ms
 + ai-edge-litert==2.1.2
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + onnx==1.20.1
 + onnx-graphsurgeon==0.5.8
 + onnx2tf==1.2

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 22 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.1s, saved as '/content/runs/detect/coco_total_yolo26n_train_fixed/weights/best.onnx' (9.5 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /content/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 61.1files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/runs/detect/coco_total_yolo26n_train_fixed/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 300, 6), dtype=tf.float32, name=None)
Captures:
  135920827009744: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  135920827007632: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  135920827008976: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  135920827013200: Ten

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image
import cv2
import os
import yaml # To load class names from data.yaml

# Define paths
tflite_model_path = '/content/best_float32.tflite' # Corrected path
image_path = '/content/100733546.s2.jpg'
output_image_path = '/content/100733546.s2_tflite_inference.jpg'
data_yaml_path = '/content/coco_total/data.yaml'

# Load class names from data.yaml
class_names = []
if os.path.exists(data_yaml_path):
    with open(data_yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)
        if 'names' in data_yaml:
            class_names = data_yaml['names']
        else:
            print(f"Warning: 'names' field not found in {data_yaml_path}. Using default class names.")
else:
    print(f"Warning: {data_yaml_path} not found. Using default class names.")

# Fallback/default class names if not found or data.yaml not present (assuming 13 classes from previous training)
if not class_names:
    class_names = [f"class_{i}" for i in range(13)]

# Load the TFLite model
try:
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()
except Exception as e:
    print(f"Error loading TFLite model: {e}")
    exit()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Get input size (assuming NCHW or NHWC format, typically 1, H, W, C or 1, C, H, W)
input_shape = input_details[0]['shape']
# Determine if NCHW or NHWC
if input_shape[1] == 3: # NCHW format (Batch, Channels, Height, Width)
    input_channels, input_height, input_width = input_shape[1], input_shape[2], input_shape[3]
    is_nchw = True
elif input_shape[3] == 3: # NHWC format (Batch, Height, Width, Channels)
    input_height, input_width, input_channels = input_shape[1], input_shape[2], input_shape[3]
    is_nchw = False
else:
    raise ValueError(f"Unsupported input shape format: {input_shape}")

# Load and preprocess the image
original_image = Image.open(image_path).convert('RGB')
original_width, original_height = original_image.size

# Resize image to model input size
image_resized = original_image.resize((input_width, input_height))
input_data = np.asarray(image_resized, dtype=np.float32)

# Normalize input data to [0, 1]
input_data = input_data / 255.0

# Expand dimensions and transpose if NCHW
if is_nchw:
    input_data = np.transpose(input_data, (2, 0, 1)) # HWC to CHW
    input_data = np.expand_dims(input_data, axis=0) # Add batch dimension NCHW
else:
    input_data = np.expand_dims(input_data, axis=0) # Add batch dimension NHWC

# Set the tensor
interpreter.set_tensor(input_details[0]['index'], input_data)

# Run inference
interpreter.invoke()

# Get output data (assuming output_data shape: (1, num_boxes, 6) -> [x1, y1, x2, y2, confidence, class_id])
output_data = interpreter.get_tensor(output_details[0]['index'])
detections = output_data[0] # Remove batch dimension

# Parameters for filtering
confidence_threshold = 0.25
iou_threshold = 0.45 # For Non-Maximum Suppression

boxes = []
scores = []
class_ids = []

for det in detections:
    conf = det[4]
    if conf > confidence_threshold:
        x1, y1, x2, y2 = det[0:4]
        class_id = int(det[5])

        # Scale normalized coordinates (relative to model input size) to original image size
        # First, scale to model input pixel values
        x1_model_scale = x1 * input_width
        y1_model_scale = y1 * input_height
        x2_model_scale = x2 * input_width
        y2_model_scale = y2 * input_height

        # Then, scale these pixel values to the original image dimensions
        scale_x = original_width / input_width
        scale_y = original_height / input_height

        x_orig_1 = int(x1_model_scale * scale_x)
        y_orig_1 = int(y1_model_scale * scale_y)
        x_orig_2 = int(x2_model_scale * scale_x)
        y_orig_2 = int(y2_model_scale * scale_y)

        # For cv2.dnn.NMSBoxes, we need [x, y, w, h]
        w_orig = x_orig_2 - x_orig_1
        h_orig = y_orig_2 - y_orig_1

        boxes.append([x_orig_1, y_orig_1, w_orig, h_orig])
        scores.append(conf)
        class_ids.append(class_id)

if not boxes:
    print("No objects detected above the confidence threshold.")
else:
    # Convert to numpy arrays
    boxes = np.array(boxes)
    scores = np.array(scores)
    class_ids = np.array(class_ids)

    # Perform Non-Maximum Suppression
    indices = cv2.dnn.NMSBoxes(boxes.tolist(), scores.tolist(), confidence_threshold, iou_threshold)

    # Convert original PIL image to OpenCV format to draw
    output_image_cv = cv2.cvtColor(np.array(original_image), cv2.COLOR_RGB2BGR)

    if len(indices) > 0:
        for i in indices.flatten():
            box = boxes[i]
            x, y, w, h = box[0], box[1], box[2], box[3]
            class_id = class_ids[i]
            score = scores[i]

            color = (0, 255, 0) # Green in BGR
            cv2.rectangle(output_image_cv, (x, y), (x + w, y + h), color, 2)
            label = f"{class_names[class_id]}: {score:.2f}"
            cv2.putText(output_image_cv, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Save the output image
    cv2.imwrite(output_image_path, output_image_cv)
    print(f"Inference result saved to {output_image_path}")

print("TFLite inference completed.")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Inference result saved to /content/100733546.s2_tflite_inference.jpg
TFLite inference completed.
